In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class FrameViT(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=16, stride=16)
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8),
            num_layers=4
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x):
        # x: [B, C, H, W]
        x = self.patch_embed(x)  # [B, D, H', W']
        
        x = x.flatten(2).transpose(1, 2)  # [B, N, D]
        
        cls_tokens = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = self.transformer(x)
        
        return x[:, 0]  # CLS token
    
class TemporalTransformer(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8),
            num_layers=4
        )

    def forward(self, x):
        # x: [B, T, D]
        x = x.transpose(0, 1)  # [T, B, D]
        x = self.transformer(x)
        return x.mean(dim=0)  # [B, D]


In [3]:
class VideoEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.frame_encoder = FrameViT(embed_dim)
        self.temporal_model = TemporalTransformer(embed_dim)

    def forward(self, video):
        # video: [B, T, C, H, W]
        B, T, C, H, W = video.shape
        
        frames = video.view(B*T, C, H, W)
        
        frame_embeddings = self.frame_encoder(frames)  # [B*T, D]
        frame_embeddings = frame_embeddings.view(B, T, -1)
        
        video_embedding = self.temporal_model(frame_embeddings)
        
        return video_embedding

In [4]:
class AudioGenerator(nn.Module):
    def __init__(self, embed_dim=256, mel_bins=80, time_steps=64):
        super().__init__()
        
        self.fc = nn.Linear(embed_dim, 512)
        
        self.net = nn.Sequential(
            nn.ConvTranspose2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            
            nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(64, 1, 3, padding=1)
        )
        
        self.mel_bins = mel_bins
        self.time_steps = time_steps

    def forward(self, z):
        x = self.fc(z)  # [B, 512]
        
        x = x.view(x.size(0), 32, 4, 4)  # reshape to feature map
        
        x = self.net(x)  # [B, 1, H, W]
        
        # Resize to desired spectrogram size
        x = F.interpolate(x, size=(self.mel_bins, self.time_steps), mode='bilinear')
        
        return x.squeeze(1)  # [B, mel_bins, time_steps]

In [5]:
class AVFootstepModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.video_encoder = VideoEncoder()
        self.audio_generator = AudioGenerator()

    def forward(self, video):
        z = self.video_encoder(video)
        spectrogram = self.audio_generator(z)
        return spectrogram

In [6]:
def loss_fn(pred, target):
    return F.l1_loss(pred, target)

In [14]:
from torch.utils.data import Dataset, DataLoader
# data from https://andrewowens.com/vis/

import os
import torch
from torch.utils.data import Dataset
import torchvision.io as io
import torchaudio

class AudioVideoDataset(Dataset):
    def __init__(self, root_dir, transform_video=None, transform_audio=None):
        self.root_dir = root_dir
        self.transform_video = transform_video
        self.transform_audio = transform_audio

        # collect base filenames (without extension)
        self.samples = sorted([
            f.replace(".mp4", "")
            for f in os.listdir(root_dir)
            if f.endswith(".mp4")
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base = self.samples[idx]

        video_path = os.path.join(self.root_dir, base + ".mp4")
        audio_path = os.path.join(self.root_dir, base + ".wav")

        # Load video (T, H, W, C)
        video = io.read_file(video_path)

        # Load audio (channels, time)
        audio, sr = torchaudio.load(audio_path)

        if self.transform_video:
            video = self.transform_video(video)

        if self.transform_audio:
            audio = self.transform_audio(audio)

        return {
            "video": video,
            "audio": audio,
            "sample_rate": sr,
            "id": base
        }

In [15]:
from torch.utils.data import DataLoader

dataset = AudioVideoDataset("vis-data-256/")
def collate_fn(batch):
    videos = [item["video"] for item in batch]
    audios = [item["audio"] for item in batch]

    # Example: pad to max length
    videos = torch.nn.utils.rnn.pad_sequence(videos, batch_first=True)
    audios = torch.nn.utils.rnn.pad_sequence(audios, batch_first=True)

    return {
        "video": videos,
        "audio": audios
    }
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)

In [17]:
model = AVFootstepModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for video, target_spec in dataloader:
    video = video.cuda()
    target_spec = target_spec.cuda()
    
    pred_spec = model(video)
    
    loss = loss_fn(pred_spec, target_spec)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print("Loss:", loss.item())

/tmp/ipykernel_114363/888580595.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(
/tmp/ipykernel_114363/888580595.py:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/ace/school/2025-2026/6.S058/cv-final-project/venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ace/school/2025-2026/6.S058/cv-final-project/venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_114363/3562390445.py", line 10, in collate_fn
    audios = torch.nn.utils.rnn.pad_sequence(audios, batch_first=True)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ace/school/2025-2026/6.S058/cv-final-project/venv/lib/python3.12/site-packages/torch/nn/utils/rnn.py", line 465, in pad_sequence
    return torch._C._nn.pad_sequence(
           ^^^^^^^^^^^^^^^^^^^^

In [ ]:
import torchaudio

waveform = torchaudio.functional.griffinlim(
    pred_spec,
    n_fft=1024
)